# Random K-Labelsets (RAkEL) - Starter Notebook
This notebook implements a simplified RAkEL-style ensemble by training label-powerset models on random subsets of labels and aggregating predictions.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_multilabel_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split

In [ ]:
def to_token(y_row):
    idx = np.where(y_row == 1)[0]
    return '|'.join(map(str, idx.tolist())) if len(idx) else 'none'


def from_token(token, k):
    out = np.zeros(k, dtype=int)
    if token != 'none':
        for t in token.split('|'):
            out[int(t)] = 1
    return out


X, Y = make_multilabel_classification(n_samples=1000, n_features=20, n_classes=8, n_labels=2, random_state=42)
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.25, random_state=42)

rng = np.random.default_rng(42)
n_labels = Y.shape[1]
subset_size = 3
n_models = 10
models = []

In [ ]:
for _ in range(n_models):
    label_subset = np.sort(rng.choice(n_labels, size=subset_size, replace=False))
    y_subset_train = Y_train[:, label_subset]
    tokens = np.array([to_token(row) for row in y_subset_train])

    model = LogisticRegression(max_iter=1200)
    model.fit(X_train, tokens)
    models.append((label_subset, model))

vote_sum = np.zeros((len(X_test), n_labels), dtype=float)
vote_count = np.zeros((len(X_test), n_labels), dtype=float)

for label_subset, model in models:
    pred_tokens = model.predict(X_test)
    decoded = np.vstack([from_token(token, len(label_subset)) for token in pred_tokens])
    for local_idx, global_label in enumerate(label_subset):
        vote_sum[:, global_label] += decoded[:, local_idx]
        vote_count[:, global_label] += 1

Y_pred = (vote_sum / np.maximum(vote_count, 1) >= 0.5).astype(int)

results = pd.DataFrame([
    {'Metric': 'Micro F1', 'Value': f1_score(Y_test, Y_pred, average='micro')},
    {'Metric': 'Macro F1', 'Value': f1_score(Y_test, Y_pred, average='macro')},
    {'Metric': 'Samples F1', 'Value': f1_score(Y_test, Y_pred, average='samples')},
]).set_index('Metric').round(3)
results